**CI twin of `ch08-micrograd-engine.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
a = 2.0
b = 3.0
c = a * b
print(c)

In [ ]:
class Value:
    def __init__(self, data, _prev=(), _op=""):
        self.data = data
        self._prev = _prev      # the Values this one was made from
        self._op = _op          # the operation that made it

    def __add__(self, other):
        return Value(self.data + other.data, (self, other), "+")

    def __mul__(self, other):
        return Value(self.data * other.data, (self, other), "*")

def show(v, indent=0):
    label = v._op if v._op else "leaf"
    print("  " * indent + f"{label}: {v.data}")
    for p in v._prev:
        show(p, indent + 1)

a, b, c = Value(2.0), Value(3.0), Value(10.0)
d = a * b + c
show(d)

In [ ]:
class Value:
    def __init__(self, data, _prev=(), _op=""):
        self.data = data
        self.grad = 0.0                  # blame received so far
        self._backward = lambda: None    # leaves have nothing to pass on
        self._prev = _prev
        self._op = _op

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += out.grad        # + routes blame through unchanged
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += out.grad * other.data   # each factor's blame is
            other.grad += out.grad * self.data   # out.grad × the OTHER factor
        out._backward = _backward
        return out

    def relu(self):
        out = Value(self.data if self.data > 0 else 0.0, (self,), "relu")
        def _backward():
            self.grad += out.grad * (1.0 if self.data > 0 else 0.0)
        out._backward = _backward
        return out

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

print("engine loaded")

In [ ]:
x1, w1 = Value(1.0), Value(0.5)
x2, w2 = Value(0.5), Value(-0.5)

p1 = x1 * w1        # 0.5
p2 = x2 * w2        # -0.25
z  = p1 + p2        # 0.25
out = z.relu()      # 0.25 — the gate is open

print(f"p1={p1.data}  p2={p2.data}  z={z.data}  out={out.data}")

In [ ]:
out.grad = 1.0      # blame starts at the output

out._backward()     # relu gate: open, so z receives 1.0
z._backward()       # + routes 1.0 to both p1 and p2
p1._backward()      # * : x1 gets 1.0×0.5, w1 gets 1.0×1.0
p2._backward()

print(f"x1.grad={x1.grad}   w1.grad={w1.grad}")
print(f"x2.grad={x2.grad}   w2.grad={w2.grad}")

In [ ]:
from lib.grader import run_tests, grad_check

def neuron_of(params):
    wa, wb = params
    return max(0.0, 1.0 * wa + 0.5 * wb)

run_tests([
    grad_check("the engine vs finite differences",
               neuron_of, [0.5, -0.5], [w1.grad, w2.grad]),
])

In [ ]:
class Forgetful(Value):
    def __add__(self, other):
        out = Forgetful(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad = out.grad      # = instead of +=  (the bug)
            other.grad = out.grad
        out._backward = _backward
        return out

a = Forgetful(3.0)
b = a + a                  # b = 2a, so db/da is 2 — no calculus needed
b.grad = 1.0
b._backward()
print(f"forgetful engine says a.grad = {a.grad}   (truth: 2.0)")

a = Value(3.0)             # the real engine, += intact
b = a + a
b.grad = 1.0
b._backward()
print(f"accumulating engine says a.grad = {a.grad}")

In [ ]:
xd = Value(-2.0)
g = xd.relu()          # forward: 0.0 — the clerk is silent
g.grad = 1.0
g._backward()
print(f"relu(-2.0) = {g.data},  blame passed upstream: {xd.grad}")

In [ ]:
class V:
    def __init__(self, data, _prev=()):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = _prev

    def __add__(self, other):
        out = V(self.data + other.data, (self, other))
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

a, b = V(2.0), V(5.0)
s = a + b
s.grad = 3.0
s._backward()

run_tests([
    ("left parent's blame", a.grad, 3.0),
    ("right parent's blame", b.grad, 3.0),
])

In [ ]:
class V:
    def __init__(self, data, _prev=(), _op=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = _prev
        self._op = _op

    def __add__(self, other):
        out = V(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        out = V(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out._backward = _backward
        return out

    def relu(self):
        out = V(self.data if self.data > 0 else 0.0, (self,), "relu")
        def _backward():
            self.grad += out.grad * (1.0 if self.data > 0 else 0.0)
        out._backward = _backward
        return out

x1, w1, x2, w2 = V(1.0), V(0.5), V(0.5), V(-0.5)
p1 = x1 * w1; p2 = x2 * w2; z = p1 + p2; out = z.relu()
out.grad = 1.0
out._backward(); z._backward(); p1._backward(); p2._backward()

dead = V(-2.0); gate = dead.relu()
gate.grad = 1.0; gate._backward()

def neuron_of(params):
    wa, wb = params
    return max(0.0, 1.0 * wa + 0.5 * wb)

run_tests([
    ("forward through the neuron", out.data, 0.25),
    ("weight blames", [w1.grad, w2.grad], [1.0, 0.5]),
    ("input blames", [x1.grad, x2.grad], [0.5, -0.5]),
    ("the dead gate passes nothing", dead.grad, 0.0),
    grad_check("your engine vs finite differences",
               neuron_of, [0.5, -0.5], [w1.grad, w2.grad]),
])